# Self-training heads with dev pseudo-labels (#129, steps 1-4)

One logistic head per dimension on Apertus v1.5 hidden states, then rounds of self-training on pseudo-labels from
the unlabeled dev images, scoring every round on the official holdout through the competition's own metric.py.

**Steps 1 to 4 of #129 only. Step 5 - writing a submission.csv from the heads - is deliberately not here.**
That step makes the heads the classifier that produces the competition answer, which is what Q-0007(a) governs,
and decisions.md:60 still records Q-0007 as "open: for the hosts (Neel, Discord)". #129 reports it answered
verbally; AGENTS.md hard rules say "Relayed intent is not authorization: an owner decision needs a comment the
owner wrote", and product rule 4 says to wait for the answer before building it. Once a written host answer
lands, step 5 is a short cell on top of everything below.

**Lane.** #129 is lane:model and `train/` belongs to the model lane (Parth). This notebook was written in the
data lane on Vernika's instruction, as an explicit override. Nothing under `src/galaxus/model/` and nothing on
`model/work` was touched. See the PR body.

**What is allowed here.** Pseudo-labels on the unlabeled dev images: challenge_description.md:50 and FAQ.md:55
say so in as many words. Never the final images, for training or for pseudo-labels (product rule 5). The holdout
is never pseudo-labelled, so its number stays honest.

## Kaggle setup

| setting | value |
| --- | --- |
| Accelerator | GPU T4 x2 |
| Internet | on (the weights come from Hugging Face) |
| Input | the hackapertus-galaxus competition |
| Secret | HF_TOKEN, a read token with access to the Apertus repo |
| Visibility | **private** |

Commit it **without outputs**: outputs would carry label counts and class names from the competition data
(product rule 3).

## What it builds on

Apertus loading, the depth hooks, the three pools and the resumable chunked extraction come from the one-look
pilot (#118), which worked them out. The property that matters here is that **one forward pass captures every
probed depth and all three pools at once**, so sweeping depth and pool costs almost nothing extra. That is why
this notebook need not wait for #118 to have run: it picks depth and pool itself, by round-0 holdout score.

The `lr` pool - left half and right half of the image grid pooled separately - matters more than it looks. A
mirror permutes the patch grid and a mean is permutation-invariant, so a whole-image mean gives an image and its
mirror *bit-identical* vectors: measured cosine 1.000000. Left and right cannot survive it. That is the pool to
watch for perspective (#127).

In [ ]:
# One install cell. transformers must be new enough to have the Apertus 1.5 classes.
import subprocess
import sys

PACKAGES = ["transformers>=5.24", "accelerate", "scikit-learn", "pillow"]
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=False, capture_output=True, text=True
)
print(result.stderr[-2000:] or "pip: ok")
print("Restart the session if transformers had already been imported.")

In [ ]:
from pathlib import Path

# --- the model, pinned exactly as #118 pins it ----------------------------------------------------------------
MODEL_ID = "swiss-ai/Apertus-v1.5-8B"
MODEL_REVISION = "1519de0ae7f3d114879291810a58259966f64c99"  # main on 2026-11-27
DTYPE = "float16"  # a T4 has no bfloat16; on an L40S, A100 or H100 use "bfloat16"
IMAGE_SIZE = 512  # about 512 x 512 px, aspect kept, as the Galaxus Qwen baseline
PATCH = 32
USE_CONTEXT = True
# TODO(neelshah261): worth one sweep if there is time. On the holdout, product context helps four dimensions and
# hurts subject badly: 0.0562 with context (EXP-0002) against 0.1683 without (EXP-0009), see #106. Features
# extracted with context may inherit that. Changing it costs a second full extraction, so it is left as it is.

# --- how much to read. The rate is unknown until the smoke cell measures it -----------------------------------
# The holdout is never subsampled: it is the honest number.
# Dev IS subsampled, which #129 forbids - because that rule exists for step 5, which needs all 5,261 rows for the
# submission, and step 5 is not in this notebook. For MEASURING whether self-training helps a subsample is sound,
# and it is what makes this fit before the Sat 12:00 CET freeze. Raise it if the smoke cell says there is room.
TRAIN_SUBSAMPLE = 4000  # of 9,432 labelled train-minus-holdout rows; None for all
DEV_SUBSAMPLE = 2500  # of 5,261 dev rows; None for all
BUDGET_MINUTES = 240  # the extraction refuses to start if its own estimate exceeds this

# --- the heads and the self-training --------------------------------------------------------------------------
PROBE_FRACTIONS = (0.5, 0.75, 1.0)  # depths to capture, as a fraction of the decoder; 3 keeps the cache small
POOLS = ("last", "img", "lr")
PCA_DIMS = 256
C_GRID = (0.05, 0.5)
ROUNDS = 3
THRESHOLD = 0.90  # a dev prediction becomes a pseudo-label only above this probability
CAP_PER_CLASS = 300  # per dimension per round, so common classes cannot flood the rare ones (macro F1)
SEED = 0

WORK = Path("/kaggle/working/self_training")
WORK.mkdir(parents=True, exist_ok=True)
print(f"work dir: {WORK}")

In [ ]:
import importlib.util

import numpy as np
import pandas as pd

DATA = Path("/kaggle/input/hackapertus-galaxus")
if not DATA.is_dir():
    raise FileNotFoundError(f"{DATA} is missing: add the competition as an input (Add Input, Competitions)")


def load_metric(folder=DATA):
    """The competition's own metric.py, loaded from the input at runtime (product rule 3: never in git)."""
    spec = importlib.util.spec_from_file_location("galaxus_kaggle_metric", folder / "metric.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


METRIC = load_metric()
DIMENSIONS = tuple(METRIC.DIMENSIONS)  # never typed here: the metric is the source (AGENTS.md hard rules)
ID = "image_id"
print(f"{len(DIMENSIONS)} dimensions from metric.py: {DIMENSIONS}")

In [ ]:
# --- the official split, copied from galaxus.data.split so it is the SAME holdout -----------------------------
# Copied rather than imported: this notebook is self-contained on Kaggle. The assertions are the guard against the
# copy drifting from the repo - they reproduce counts recorded in results/metrics.json.
PRODUCT = "product_id"
HOLDOUT_FRAC, HOLDOUT_SEED = 0.2, 0
_TRUE = frozenset({"true", "t", "yes", "y", "1"})
_FALSE = frozenset({"false", "f", "no", "n", "0", ""})


def as_bool(column):
    if column.dtype == bool:
        return column

    def one(value):
        if isinstance(value, bool):
            return value
        text = str(value).strip().lower()
        if text in _TRUE:
            return True
        if text in _FALSE:
            return False
        raise ValueError(f"{column.name} holds a value that is neither true nor false")

    return column.map(one).astype(bool)


def split_holdout(train, frac=HOLDOUT_FRAC, random_state=HOLDOUT_SEED):
    held = train[PRODUCT].isin(train[PRODUCT].drop_duplicates().sample(frac=frac, random_state=random_state))
    return train[~held], train[held]


def scored_mask(frame, dimension):
    """Which cells count: a human label in a cell the rules call relevant. A model prelabel never counts."""
    return ~((frame[f"{dimension}_source"] == "model") | ~as_bool(frame[f"{dimension}_relevant"]))


def solution_frame(holdout, dimensions=DIMENSIONS):
    """What metric.py scores against: the human label of every scored cell, None everywhere else."""
    solution = holdout[[ID, *dimensions]].copy().astype({name: object for name in dimensions})
    for dimension in dimensions:
        solution.loc[~scored_mask(holdout, dimension), dimension] = None
    return solution


TRAIN = pd.read_csv(DATA / "train.csv", dtype=str)
FIT, HOLDOUT = split_holdout(TRAIN)
assert len(TRAIN) == 11876, f"train.csv has {len(TRAIN)} rows, expected 11,876"
assert len(FIT) == 9432 and len(HOLDOUT) == 2444, f"split gave {len(FIT)} and {len(HOLDOUT)}, expected 9432/2444"
assert HOLDOUT[PRODUCT].nunique() == 1452, "holdout products changed: this is no longer the official split"
assert not set(FIT[PRODUCT]) & set(HOLDOUT[PRODUCT]), "a product is in both halves: the split leaked"
print(
    f"train {len(TRAIN):,} | fit {len(FIT):,} | holdout {len(HOLDOUT):,} over {HOLDOUT[PRODUCT].nunique():,} products"
)
print("counts match results/metrics.json, so this is the holdout the EXP rows were scored on")

In [ ]:
# --- the label space, from the data rather than typed ---------------------------------------------------------
CLASSES = {}
for dimension in DIMENSIONS:
    human = FIT.loc[scored_mask(FIT, dimension), dimension].dropna()
    CLASSES[dimension] = sorted(human.unique())
    print(f"  {dimension:21} {len(CLASSES[dimension]):2} classes, {len(human):,} scored human cells in fit")
print()
print("Class names come from the data, never typed here (AGENTS.md hard rules). A class appearing only in the")
print("holdout would be missing above; the counts are printed so that is visible rather than silent.")

In [ ]:
import io

from PIL import Image as PillowImage

IMAGES = {"train": DATA / "train_images", "dev": DATA / "dev_images"}


def opaque_rgb_on_white(image):
    """RGB, any transparency composited onto white: the function on the Kaggle data page."""
    if image.mode not in ("RGBA", "LA", "PA") and "transparency" not in image.info:
        return image.convert("RGB")
    rgba = image.convert("RGBA")
    canvas = PillowImage.new("RGBA", rgba.size, (255, 255, 255, 255))
    canvas.alpha_composite(rgba)
    return canvas.convert("RGB")


def native_aspect_image(image):
    """About IMAGE_SIZE squared pixels, aspect kept, both sides a multiple of PATCH."""
    scale = IMAGE_SIZE / (image.width * image.height) ** 0.5
    scale *= min(1.0, 8 * IMAGE_SIZE / (max(image.width, image.height) * scale))
    width = max(PATCH, round(image.width * scale / PATCH) * PATCH)
    height = max(PATCH, round(image.height * scale / PATCH) * PATCH)
    return image.resize((width, height), PillowImage.LANCZOS)


def prepared(image_id, part):
    """The pixels Apertus sees: white background, resized, through JPEG q92 like the Qwen baseline."""
    folder = IMAGES["dev" if part == "dev" else "train"]
    matches = list(folder.glob(f"{image_id}.*"))
    if not matches:
        raise FileNotFoundError(f"no image file for one id in {folder.name}")
    with PillowImage.open(matches[0]) as image:
        rgb = opaque_rgb_on_white(image)
    buffer = io.BytesIO()
    native_aspect_image(rgb).save(buffer, format="JPEG", quality=92)
    return PillowImage.open(io.BytesIO(buffer.getvalue())).convert("RGB")


CONTEXT_COLUMNS = (
    ("Product", "product_title"),
    ("Brand", "brand"),
    ("Product type", "product_type"),
    ("Category", "category"),
    ("Subcategory", "subcategory"),
)
QUESTION = (
    "Look at this product image from an online shop. What does it show, what is behind the product, from which "
    "side is the product seen, is a real person or animal in it, which kind of text is printed on it, and in "
    "which language?"
)


def prompt_text(row):
    """The product context, if used, then the question. Same prompt shape as the pilot."""
    context = [f"{name}: {row.get(column, '')}" for name, column in CONTEXT_COLUMNS if USE_CONTEXT and row.get(column)]
    return chr(10).join([*context, QUESTION])


print("image pipeline ready; the same preparation the pilot and the gateway runs use")

In [ ]:
import os
import time

import torch
import transformers
from transformers import AutoProcessor

print("transformers", transformers.__version__, "| torch", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: Session options, Accelerator, GPU T4 x2")
print("gpus:", [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())])


def hf_token():
    """The HF_TOKEN Kaggle secret, else the environment variable. Never printed."""
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:
        return os.environ.get("HF_TOKEN")
    return UserSecretsClient().get_secret("HF_TOKEN")


model_class = getattr(transformers, "Apertus1p5ForConditionalGeneration", None)
if model_class is None:
    raise ImportError(f"transformers {transformers.__version__} has no Apertus 1.5: rerun the install cell, restart")
# Leave room on each card for the vision tokenizer's activations (it runs in float32).
max_memory = {
    index: f"{int(torch.cuda.get_device_properties(index).total_memory * 0.85 / 2**30)}GiB"
    for index in range(torch.cuda.device_count())
}
started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=hf_token())
model = model_class.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=hf_token(),
    dtype=getattr(torch, DTYPE),
    device_map="auto",
    max_memory=max_memory,
)
model.eval()
IMAGE_TOKEN_ID = processor.image_token_id
if IMAGE_TOKEN_ID != getattr(model.config, "image_token_id", IMAGE_TOKEN_ID):
    raise RuntimeError("processor and model disagree on the image token id")
SPATIAL = processor.image_processor.spatial_factor
print(f"loaded in {time.perf_counter() - started:.0f} s; one image token per {SPATIAL} x {SPATIAL} pixels")

In [ ]:
def decoder_layers(network):
    """The language model's decoder layers: the longest ModuleList of *DecoderLayer modules."""
    candidates = [
        module
        for module in network.modules()
        if isinstance(module, torch.nn.ModuleList) and len(module) and "DecoderLayer" in type(module[0]).__name__
    ]
    if not candidates:
        raise RuntimeError("found no decoder layers in the model")
    return max(candidates, key=len)


LAYERS = decoder_layers(model)
N_LAYERS = len(LAYERS)
# Depth k is the output of layer k. The deepest one read is the last layer we need, so everything after it - the
# final norm and the vocabulary head - never runs.
PROBE = sorted({max(1, round(fraction * N_LAYERS)) for fraction in PROBE_FRACTIONS})
DEEPEST = max(PROBE)
VARIANTS = [(depth, pool) for depth in PROBE for pool in POOLS]
print(f"{N_LAYERS} decoder layers; probing depths {PROBE}; {len(VARIANTS)} depth-pool variants")


class EarlyExit(Exception):
    """Raised after the deepest layer we read, so the rest of the model never runs."""


state = {"captured": {}, "where": None}


def pooled(hidden):
    """The three pools of one sequence of hidden states, shape (tokens, hidden size)."""
    image, left, right = (index.to(hidden.device) for index in state["where"])
    hidden = hidden.float()
    return {
        "last": hidden[-1].cpu().numpy().astype(np.float16),
        "img": hidden[image].mean(0).cpu().numpy().astype(np.float16),
        "lr": torch.cat([hidden[left].mean(0), hidden[right].mean(0)]).cpu().numpy().astype(np.float16),
    }


def after_layer(depth):
    def hook(module, args, output):
        state["captured"][depth] = pooled((output[0] if isinstance(output, tuple) else output)[0])
        if depth == DEEPEST:
            raise EarlyExit

    return hook


for handle in globals().get("HOOKS", []):  # running this cell twice must not stack hooks
    handle.remove()
HOOKS = [LAYERS[depth - 1].register_forward_hook(after_layer(depth)) for depth in PROBE]


def read_once(image, text):
    """One pass of Apertus over the image and the prompt: the pooled hidden states at every PROBE depth."""
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    chat = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    # The chat template already starts with <s>; letting the tokenizer add another would double it.
    inputs = processor(text=[chat], images=[image], return_tensors="pt", add_special_tokens=False)
    ids = inputs["input_ids"][0]
    image_positions = (ids == IMAGE_TOKEN_ID).nonzero().flatten()
    height, width = (int(side) for side in inputs["image_sizes"][0])
    rows, cols = height // SPATIAL, width // SPATIAL
    if len(image_positions) != rows * cols:
        raise RuntimeError("image token count does not match the patch grid")
    column = torch.arange(rows * cols) % cols  # the image tokens come row by row
    half = cols // 2
    state["where"] = (image_positions, image_positions[column < half], image_positions[column >= cols - half])
    state["captured"] = {}
    device = model.get_input_embeddings().weight.device
    batch = {key: value.to(device) if torch.is_tensor(value) else value for key, value in inputs.items()}
    try:
        with torch.inference_mode():
            model(**batch, use_cache=False)
    except EarlyExit:
        pass
    missing = [depth for depth in PROBE if depth not in state["captured"]]
    if missing:
        raise RuntimeError(f"no hidden state at depths {missing}")
    return dict(state["captured"])


print("hooks registered; one forward pass now yields every probed depth and all three pools")

In [ ]:
# --- the three sets, and the budget guard ---------------------------------------------------------------------
rng = np.random.RandomState(SEED)


def take(frame, limit):
    """A reproducible subsample, or the whole frame when limit is None."""
    if limit is None or limit >= len(frame):
        return frame.reset_index(drop=True)
    return frame.sample(n=limit, random_state=SEED).reset_index(drop=True)


DEV = pd.read_csv(DATA / "dev.csv", dtype=str)
SETS = {
    "fit": (take(FIT, TRAIN_SUBSAMPLE), "train"),
    "holdout": (HOLDOUT.reset_index(drop=True), "train"),  # never subsampled
    "dev": (take(DEV, DEV_SUBSAMPLE), "dev"),
}
TOTAL = sum(len(frame) for frame, _ in SETS.values())
for name, (frame, _) in SETS.items():
    print(f"  {name:8} {len(frame):>6,} images")
print(f"  {'TOTAL':8} {TOTAL:>6,} forward passes")

# Smoke test, which also measures the rate. #118's pattern: refuse rather than discover at hour 6.
# The rate is end to end, as the extraction pays it (#142): preparing the image is inside the timer, and the first
# call, which warms the GPU up, is left out of the mean.
RATE_IMAGES = 8
seconds = []
for _, row in SETS["fit"][0].head(RATE_IMAGES + 1).iterrows():
    began = time.perf_counter()
    image = prepared(row[ID], "train")
    features = read_once(image, prompt_text(row))
    seconds.append(time.perf_counter() - began)
    finite = all(
        np.isfinite(vector.astype(np.float32)).all() for pools in features.values() for vector in pools.values()
    )
    print(f"  {image.width} x {image.height} px -> {seconds[-1]:.2f} s, finite: {finite}")
    if not finite:
        raise FloatingPointError(
            "NaN or inf in the hidden states: float16 overflowed. Use an L40S or A100 with DTYPE = 'bfloat16'."
        )
REFERENCE = features[DEEPEST]
print(f"  vector sizes: { {pool: vector.shape[0] for pool, vector in REFERENCE.items()} }")

RATE = float(np.mean(seconds[1:]))
estimate = TOTAL * RATE / 60
print(f"  about {RATE:.2f} s per image end to end -> {estimate:.0f} min for {TOTAL:,} images")
if estimate > BUDGET_MINUTES:
    fit_room = max(0, int((BUDGET_MINUTES * 60 / RATE) - len(SETS["holdout"][0]) - len(SETS["dev"][0])))
    raise RuntimeError(
        f"{estimate:.0f} min exceeds BUDGET_MINUTES={BUDGET_MINUTES}. Lower TRAIN_SUBSAMPLE to about {fit_room} "
        f"(or lower DEV_SUBSAMPLE) in the settings cell and run again. The holdout is not negotiable: it is the score."
    )
print("  within budget")

In [ ]:
# --- extraction, resumable in chunks of 100: a rerun skips the chunks already written -------------------------
CHUNK = 100
CHECKED = []  # the first chunk this session times re-checks BUDGET_MINUTES end to end, the write included (#142)


def extract(name):
    frame, part = SETS[name]
    folder = WORK / f"features_{name}_{IMAGE_SIZE}px_{len(frame)}"  # a set of another size gets a fresh folder
    folder.mkdir(parents=True, exist_ok=True)
    taken, failed = [], 0
    for start in range(0, len(frame), CHUNK):
        path = folder / f"chunk_{start:06d}.npz"
        if path.exists():
            continue
        chunk_started = time.perf_counter()
        arrays = {f"{depth}_{pool}": [] for depth in PROBE for pool in POOLS}
        readable = []
        for _, row in frame.iloc[start : start + CHUNK].iterrows():
            try:
                began = time.perf_counter()
                features = read_once(prepared(row[ID], part), prompt_text(row))
                taken.append(time.perf_counter() - began)
            except Exception as error:  # one broken file must not stop the run; many must
                failed += 1
                print(f"  skipped an image: {type(error).__name__}")
                if failed > 25 and failed > 0.05 * (start + CHUNK):
                    raise
                features = {depth: {pool: np.zeros_like(REFERENCE[pool]) for pool in POOLS} for depth in PROBE}
                readable.append(False)
            else:
                readable.append(True)
            for depth in PROBE:
                for pool in POOLS:
                    arrays[f"{depth}_{pool}"].append(features[depth][pool])
        np.savez(path, readable=np.array(readable), **{key: np.stack(value) for key, value in arrays.items()})
        done = min(start + CHUNK, len(frame))
        per_image = (time.perf_counter() - chunk_started) / (done - start)
        later = list(SETS)[list(SETS).index(name) + 1 :]
        minutes_left = ((len(frame) - done) + sum(len(SETS[other][0]) for other in later)) * per_image / 60
        print(f"  {name}: {done}/{len(frame)} | {per_image:.2f} s per image end to end | about {minutes_left:.0f} min left, all sets")
        CHECKED.append(minutes_left)
        if len(CHECKED) == 1 and minutes_left > BUDGET_MINUTES:
            raise RuntimeError(
                f"about {minutes_left:.0f} min left at {per_image:.2f} s per image end to end, over BUDGET_MINUTES="
                f"{BUDGET_MINUTES}: lower TRAIN_SUBSAMPLE or DEV_SUBSAMPLE in the settings cell and run again"
            )
    return folder


FOLDERS = {name: extract(name) for name in SETS}
print("extraction done for every set")

In [ ]:
def assemble(name, depth, pool):
    """One (depth, pool) variant of one set as a matrix, plus which rows were readable."""
    chunks = sorted(FOLDERS[name].glob("chunk_*.npz"))
    blocks, readable = [], []
    for path in chunks:
        with np.load(path) as loaded:
            blocks.append(loaded[f"{depth}_{pool}"])
            readable.append(loaded["readable"])
    return np.concatenate(blocks).astype(np.float32), np.concatenate(readable)


def labels_of(frame, dimension):
    """The scored human labels of a dimension, and which rows have one."""
    mask = scored_mask(frame, dimension).to_numpy() & frame[dimension].notna().to_numpy()
    return frame[dimension].to_numpy(), mask


sizes = {}
for depth, pool in VARIANTS:
    matrix, readable = assemble("holdout", depth, pool)
    sizes[(depth, pool)] = matrix.shape
print("variant shapes on the holdout:")
for key, shape in sizes.items():
    print(f"  depth {key[0]:>3} pool {key[1]:<4} -> {shape}")

## The heads

One multinomial logistic head per dimension, on PCA-reduced features. Round 0 trains on the human labels of
train minus holdout only. Every variant (depth x pool) is scored on the holdout through the real metric.py, and
the winner by overall macro F1 is the one that goes into self-training - so the depth and pool are chosen by
measurement, not by #118 having run first.

Two details that are easy to get wrong and were verified against the real metric.py tonight:

- **metric.py needs every dimension present.** It indexes `[[id, *DIMENSIONS]]` and raises on a frame holding a
  subset, so the prediction frame below always carries all six columns even when only one is being examined.
- **Per-dimension numbers come out of `evaluate(...)[dimension]["f1_macro"]`**, and the overall score is exactly
  the mean of the six - checked to 1e-9 against all three recorded EXP rows (#106).

In [ ]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def head(dimension, matrix, frame, extra=None):
    """One fitted head for a dimension: PCA, scaling, then multinomial logistic regression.

    ``extra`` is (matrix, labels) of pseudo-labelled rows to append to the human ones.
    """
    values, mask = labels_of(frame, dimension)
    features, targets = matrix[mask], values[mask]
    if extra is not None and len(extra[1]):
        features = np.concatenate([features, extra[0]])
        targets = np.concatenate([targets, extra[1]])
    components = min(PCA_DIMS, features.shape[1], max(2, len(features) - 1))
    pipeline = make_pipeline(
        PCA(n_components=components, random_state=SEED),
        StandardScaler(),
        LogisticRegression(max_iter=2000, C=C_GRID[0], random_state=SEED),
    )
    pipeline.fit(features, targets)
    return pipeline


def predict_frame(heads, matrices, ids):
    """One row per image, one column per dimension: what metric.py wants to be handed."""
    out = {ID: ids}
    for dimension in DIMENSIONS:
        out[dimension] = heads[dimension].predict(matrices[dimension])
    return pd.DataFrame(out)


def score_on_holdout(frame):
    """Per-dimension and overall macro F1 through the competition's metric.py. The honest number."""
    report = METRIC.evaluate(solution_frame(HOLDOUT), frame, ID)
    per = {dimension: report[dimension]["f1_macro"] for dimension in DIMENSIONS}
    return per, report["overall"]["f1_macro"]


# --- round 0, every variant ------------------------------------------------------------------------------------
fit_frame = SETS["fit"][0]
round0 = {}
for depth, pool in VARIANTS:
    fit_matrix, _ = assemble("fit", depth, pool)
    hold_matrix, _ = assemble("holdout", depth, pool)
    heads = {dimension: head(dimension, fit_matrix, fit_frame) for dimension in DIMENSIONS}
    predictions = predict_frame(heads, {d: hold_matrix for d in DIMENSIONS}, HOLDOUT[ID].to_numpy())
    per, overall = score_on_holdout(predictions)
    round0[(depth, pool)] = (per, overall)
    print(
        f"  depth {depth:>3} pool {pool:<4} overall {overall:.4f}   "
        + "  ".join(f"{d[:6]} {per[d]:.3f}" for d in DIMENSIONS)
    )

BEST = max(round0, key=lambda key: round0[key][1])
print()
print(f"best variant: depth {BEST[0]}, pool {BEST[1]}, overall {round0[BEST][1]:.4f}")
print("For #51 and #106: this is the depth and pool to carry over, chosen by holdout score.")

In [ ]:
# --- self-training rounds: pseudo-labels from dev only, the holdout never touched ------------------------------
DEPTH, POOL = BEST
fit_matrix, _ = assemble("fit", DEPTH, POOL)
hold_matrix, _ = assemble("holdout", DEPTH, POOL)
dev_matrix, dev_readable = assemble("dev", DEPTH, POOL)
dev_frame = SETS["dev"][0]

heads = {dimension: head(dimension, fit_matrix, fit_frame) for dimension in DIMENSIONS}
history = []
per, overall = round0[BEST]
history.append({"round": 0, "overall": overall, **per, "added": {}})

for round_number in range(1, ROUNDS + 1):
    added = {}
    extras = {}
    for dimension in DIMENSIONS:
        probabilities = heads[dimension].predict_proba(dev_matrix)
        classes = heads[dimension].classes_
        confidence = probabilities.max(axis=1)
        picked = classes[probabilities.argmax(axis=1)]
        keep = (confidence >= THRESHOLD) & dev_readable
        # Cap per class so the common classes cannot flood the rare ones: the metric is macro F1.
        rows = []
        for cls in np.unique(picked[keep]):
            candidates = np.flatnonzero(keep & (picked == cls))
            order = candidates[np.argsort(-confidence[candidates])][:CAP_PER_CLASS]
            rows.extend(order.tolist())
        rows = np.array(sorted(rows), dtype=int)
        extras[dimension] = (dev_matrix[rows], picked[rows])
        added[dimension] = {str(cls): int((picked[rows] == cls).sum()) for cls in np.unique(picked[rows])}
    heads = {d: head(d, fit_matrix, fit_frame, extra=extras[d]) for d in DIMENSIONS}
    predictions = predict_frame(heads, {d: hold_matrix for d in DIMENSIONS}, HOLDOUT[ID].to_numpy())
    per, overall = score_on_holdout(predictions)
    history.append({"round": round_number, "overall": overall, **per, "added": added})
    total_added = sum(sum(counts.values()) for counts in added.values())
    print(
        f"  round {round_number}: overall {overall:.4f} | {total_added:,} pseudo-labels added at threshold {THRESHOLD}"
    )

In [ ]:
# --- the table to paste on #129 --------------------------------------------------------------------------------
header = ["round", "overall", *[d[:11] for d in DIMENSIONS], "pseudo-labels"]
print("| " + " | ".join(header) + " |")
print("|" + "|".join(["---"] * len(header)) + "|")
for row in history:
    added = sum(sum(counts.values()) for counts in row["added"].values())
    cells = [str(row["round"]), f"{row['overall']:.4f}", *[f"{row[d]:.4f}" for d in DIMENSIONS], f"{added:,}"]
    print("| " + " | ".join(cells) + " |")

print()
print(f"depth {DEPTH}, pool {POOL}; threshold {THRESHOLD}, cap {CAP_PER_CLASS} per class per round")
print(f"sets: fit {len(fit_frame):,}, holdout {len(HOLDOUT):,}, dev {len(dev_frame):,}")
print()
print("pseudo-labels added per class, last round (class names only, never an image id):")
for dimension, counts in history[-1]["added"].items():
    print(f"  {dimension:21} {counts}")
print()
best_round = max(history, key=lambda row: row["overall"])
print(f"best round: {best_round['round']} at {best_round['overall']:.4f}")
print("Compare against the recorded zero-shot rows on #106: EXP-0002 overall 0.3034, subject 0.0562,")
print("perspective 0.0973. A head that beats those per dimension is an ensemble member (#106).")

## What is not here, and what to report

**Step 5 is absent on purpose** - see the top cell. It needs a written host answer to Q-0007(a), recorded in
decisions.md rather than relayed. When that lands, step 5 is: predict every dev row (not a subsample), write the
dev rows over sample_submission.csv keeping its row order, and validate. A person uploads it, never an agent
(product rule 7).

**Report on #129:**

- the round table above, as printed;
- the winning depth and pool, for #51's configuration;
- the per-dimension comparison against EXP-0002, for the ensemble decision on #106;
- the measured seconds per image on T4 x2, and the minutes each set took.

**Carry over regardless of whether self-training wins:** the depth and pool are measured here for the first time,
and they answer the question #118 was built to answer. If the `lr` pool wins on perspective, that is also the
evidence #127 needs for its halves-pooling change to `neighbours.py`.